In [1]:
import pandas as pd
import networkx as nx
import os
import numpy as np
from collections import deque
from gene_to_uniprot import convert_gene_list#genes--uniprot

DATA_PATH = r"C:\Users\Nisrin Fariss Lamine\Downloads\tfm"

def cargar_datos():
    print("Cargando BioGRID...")
    df_edges = pd.read_csv(os.path.join(DATA_PATH, "biogrid_edges.csv"))
    G = nx.from_pandas_edgelist(df_edges, source="source", target="target")

    print("Cargando DrugBank limpio...")
    df_drug = pd.read_csv(os.path.join(DATA_PATH, "drugbank_targets_clean.csv"))

    print("Cargando Enfermedades...")
    df_disorders = pd.read_csv(os.path.join(DATA_PATH, "disorder_genes.csv"), sep=";")

    return G, df_drug, df_disorders
def bfs_multifuente(grafo, origenes):
    
    distancias = {nodo: float("inf") for nodo in grafo.nodes()}#dist inf,nodos no visitados
    cola = deque()

    for o in origenes:
        if o in distancias:#si esta en nodos no visitados
            distancias[o] = 0# a simismo 
            cola.append(o)

    while cola:
        actual = cola.popleft()
        for vecino in grafo.neighbors(actual):
            if distancias[vecino] == float("inf"):
                distancias[vecino] = distancias[actual] + 1
                cola.append(vecino)

    return distancias#diccionario de nodo y su distancia minima desde  laenfermedad al resto de nodos

def construir_bins_por_grado(grafo, tamaño_bin=50):
    
    grados = {nodo: grafo.degree(nodo) for nodo in grafo.nodes()}#conexion de cada nodo
    bins = {}
    for proteina, grado in grados.items():
        bin_id = grado // tamaño_bin #ver a q bin corresponde, eneteros
        bins.setdefault(bin_id, []).append(proteina)
        
    return grados, bins
def seleccionar_enfermedad_valida(df_disorders, G):
    for enf in df_disorders["disorder"].unique():

        genes_str = df_disorders[df_disorders["disorder"] == enf]["gene_symb"].values[0]
        genes_simbolos = [g.strip() for g in genes_str.split(",") if g.strip()]

        conversion = convert_gene_list(genes_simbolos)

        genes_uniprot = [
            u
            for lista in conversion.values()
            for u in lista
        ]

        genes_presentes = [
            u for u in genes_uniprot
            if u in G.nodes()
        ]

        if len(genes_presentes) >= 1:
            return enf, genes_presentes

    return None, []
def generar_dianas_aleatorias(dianas_reales, grados, bins, tamaño_bin=50):
    
    aleatorias = []

    for diana in dianas_reales:

        grado = grados.get(diana)#grado de esa diana

        if grado is None:# si no esta en el grafo se ignora
            continue

        bin_id = grado // tamaño_bin

        if bin_id not in bins:#bin inexistente se ignora
            continue
        aleatorias.append(random.choice(bins[bin_id]))#selecciona una proteína aleatoria del mismo bin


    return aleatorias

def distancia_media_conjunto(dianas, distancias):
    
    #la distancia de cada diana desde el diccionario de distancias
    valores = [distancias.get(d, float("inf")) for d in dianas]

    # no alcanzables(infinito)
    valores = [v for v in valores if v != float("inf")]

    return sum(valores) / len(valores) if valores else None

def proximidad_estadistica(dianas, distancias_ref, grados, bins, repeticiones=200):
    dist_obs = distancia_media_conjunto(dianas, distancias_ref)#dist real

    if dist_obs is None:
        return None, None, None, None, None

    dist_aleatorias = []

    for _ in range(repeticiones): #simulaciones =

        d_rand = generar_dianas_aleatorias(dianas, grados, bins)

        dist_rand = distancia_media_conjunto(d_rand, distancias_ref)
        if dist_rand is not None:
            dist_aleatorias.append(dist_rand)

    if len(dist_aleatorias) < 3: #en caso de pocas observaciones
        return dist_obs, None, None, None, None

    media = np.mean(dist_aleatorias)
    desviacion = np.std(dist_aleatorias) if np.std(dist_aleatorias) > 0 else 1e-9#q no divida entre 0

    z = (dist_obs - media) / desviacion#  cuántas desviaciones está la real respecto a la media
    p = (1 + sum(x <= dist_obs for x in dist_aleatorias)) / (1 + len(dist_aleatorias))

    return dist_obs, media, desviacion, z, p

    

In [7]:
np.version


<module 'numpy.version' from 'C:\\Users\\Nisrin Fariss Lamine\\anaconda3\\Lib\\site-packages\\numpy\\version.py'>

In [2]:
def bfs_multifuente(grafo, origenes):
    
    distancias = {nodo: float("inf") for nodo in grafo.nodes()}#dist inf,nodos no visitados
    cola = deque()

    for o in origenes:
        if o in distancias:#si esta en nodos no visitados
            distancias[o] = 0# a simismo 
            cola.append(o)

    while cola:
        actual = cola.popleft()
        for vecino in grafo.neighbors(actual):
            if distancias[vecino] == float("inf"):
                distancias[vecino] = distancias[actual] + 1
                cola.append(vecino)

    return distancias#diccionario de nodo y su distancia minima desde  laenfermedad al resto de nodos


In [3]:
G = nx.Graph()
G.add_edges_from([
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "E")
])
origenes = ["A"]

dist = bfs_multifuente(G, origenes)

print(dist)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 1}


In [3]:
def construir_bins_por_grado(grafo, tamaño_bin=50):
    
    grados = {nodo: grafo.degree(nodo) for nodo in grafo.nodes()}#conexion de cada nodo
    bins = {}
    for proteina, grado in grados.items():
        bin_id = grado // tamaño_bin #ver a q bin corresponde, eneteros
        bins.setdefault(bin_id, []).append(proteina)
        
    return grados, bins

In [5]:
grados, bins = construir_bins_por_grado(G, tamaño_bin=2)

print("Grados:", grados)
print("Bins:", bins)

Grados: {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1}
Bins: {1: ['A', 'B', 'C'], 0: ['D', 'E']}


In [6]:
def seleccionar_enfermedad_valida(df_disorders, G):
    for enf in df_disorders["disorder"].unique():

        genes_str = df_disorders[df_disorders["disorder"] == enf]["gene_symb"].values[0]
        genes_simbolos = [g.strip() for g in genes_str.split(",") if g.strip()]

        conversion = convert_gene_list(genes_simbolos)

        genes_uniprot = [
            u
            for lista in conversion.values()
            for u in lista
        ]

        genes_presentes = [
            u for u in genes_uniprot
            if u in G.nodes()
        ]

        if len(genes_presentes) >= 1:
            return enf, genes_presentes

    return None, []


In [8]:
grados, bins = construir_bins_por_grado(G)

In [9]:
def generar_dianas_aleatorias(dianas_reales, grados, bins, tamaño_bin=50):
    
    aleatorias = []

    for diana in dianas_reales:

        grado = grados.get(diana)#grado de esa diana

        if grado is None:# si no esta en el grafo se ignora
            continue

        bin_id = grado // tamaño_bin

        if bin_id not in bins:#bin inexistente se ignora
            continue
        aleatorias.append(random.choice(bins[bin_id]))#selecciona una proteína aleatoria del mismo bin


    return aleatorias

In [10]:

import random
# coger un fármaco real
drug = df_drug["DrugBank_ID"].iloc[0]

dianas_reales = list(
    set(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])
)

print("Dianas reales:", dianas_reales[:5])

# ejecutar función
aleatorias = generar_dianas_aleatorias(dianas_reales, grados, bins)

print("Dianas aleatorias:", aleatorias[:5])

Dianas reales: ['P00734']
Dianas aleatorias: ['Q8NGE9']


In [11]:
print(len(dianas_reales), len(aleatorias))

1 1


In [12]:
for real, rand in zip(dianas_reales[:5], aleatorias[:5]):
    print(f"{real} ({grados.get(real)}) → {rand} ({grados.get(rand)})")#mismo bin

P00734 (41) → Q8NGE9 (4)


In [13]:
def distancia_media_conjunto(dianas, distancias):
    
    #la distancia de cada diana desde el diccionario de distancias
    valores = [distancias.get(d, float("inf")) for d in dianas]

    # no alcanzables(infinito)
    valores = [v for v in valores if v != float("inf")]

    return sum(valores) / len(valores) if valores else None




In [14]:

# distancias desde enfermedad
dist_ref = bfs_multifuente(G, genes)

# coger un fármaco
drug = df_drug["DrugBank_ID"].iloc[0]

dianas = list(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])

# calcular media
media = distancia_media_conjunto(dianas, dist_ref)

print("Distancia media:", media)

Distancia media: 2.0


In [15]:
print(set(dist_ref.keys()) == set(G.nodes()))# todos los nodos del grafo

True


In [16]:
def proximidad_estadistica(dianas, distancias_ref, grados, bins, repeticiones=200):
    dist_obs = distancia_media_conjunto(dianas, distancias_ref)#dist real

    if dist_obs is None:
        return None, None, None, None, None

    dist_aleatorias = []

    for _ in range(repeticiones): #simulaciones =

        d_rand = generar_dianas_aleatorias(dianas, grados, bins)

        dist_rand = distancia_media_conjunto(d_rand, distancias_ref)
        if dist_rand is not None:
            dist_aleatorias.append(dist_rand)

    if len(dist_aleatorias) < 3: #en caso de pocas observaciones
        return dist_obs, None, None, None, None

    media = np.mean(dist_aleatorias)
    desviacion = np.std(dist_aleatorias) if np.std(dist_aleatorias) > 0 else 1e-9#q no divida entre 0

    z = (dist_obs - media) / desviacion#  cuántas desviaciones está la real respecto a la media
    p = (1 + sum(x <= dist_obs for x in dist_aleatorias)) / (1 + len(dist_aleatorias))

    return dist_obs, media, desviacion, z, p

In [17]:
print("Enfermedad:", enf)
print("Genes:", genes)

Enfermedad: 3-M syndrome
Genes: ['Q14999', 'Q14999', 'O75147', 'Q9H0W5', 'Q9H0W5']


In [18]:
for drug in df_drug["DrugBank_ID"].unique():
    dianas = list(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])
    dianas_en_red = [d for d in dianas if d in G.nodes()]
    if len(dianas_en_red) >= 1:
        break

print("\nFármaco:", drug)
print("Dianas en red:", dianas_en_red)


Fármaco: DB00001
Dianas en red: ['P00734']


In [19]:

resultado = proximidad_estadistica(dianas_en_red,dist_ref,grados,bins,repeticiones=200 )

print("dist_obs:", resultado[0])
print("media:", resultado[1])
print("std:", resultado[2])
print("z:", resultado[3])
print("p:", resultado[4])

dist_obs: 2.0
media: 2.115
std: 0.4022126303337577
z: -0.2859184205741444
p: 0.8756218905472637


 cerca de lo esperado (z negativo) pero p?

In [ ]:
print(len(dianas))

In [25]:
if __name__ == "__main__":
    G, df_drug, df_disorders = cargar_datos()

    grados, bins = construir_bins_por_grado(G)

    enfermedad = "Duane retraction syndrome"  # CAMBIA AQUÍ el nombre exacto de la enfermedad(mirar csv)

    if enfermedad not in df_disorders["disorder"].values:
        print(f"\nERROR: La enfermedad '{enfermedad}' no existe en el CSV.")
        print("\nEnfermedades disponibles:")
        for e in df_disorders["disorder"].dropna().unique()[:50]:
            print("-", e)
        exit()


    # genes de la enfermedad
    genes_str = df_disorders[df_disorders["disorder"] == enfermedad]["gene_symb"].values[0]
    genes_simbolos = [g.strip() for g in genes_str.split(",") if g.strip()]

    print(f"Genes en el CSV: {genes_simbolos}")
    #conv ggenes--> UniProt
    conversion = convert_gene_list(genes_simbolos)
    
    genes_uniprot = [
        u
        for lista in conversion.values()
        for u in lista
    ]
    
    # filtrar si estan en el interactoma
    genes_validos = [
        u for u in genes_uniprot
        if u in G.nodes()
    ]
    
    print(f"Genes convertidos a UniProt: {len(genes_uniprot)}")
    print(f"Proteínas de enfermedad presentes en red: {len(genes_validos)}")
    print(f"Proteínas válidas: {genes_validos}")

    if len(genes_validos) == 0:
        print("\nERROR: Ningún gen de esta enfermedad está presente en el interactoma.")
        exit()

    #distancias 
    dist_ref = bfs_multifuente(G, genes_validos)
    
    #fármaco    
    #ejemplo_farmaco = df_drug["DrugBank_ID"].iloc[0] primero
    #ejemplo_farmaco = "DB00157" 
    ejemplo_farmaco = df_drug["DrugBank_ID"].sample(1).values[0]#aleatorio

    if ejemplo_farmaco not in df_drug["DrugBank_ID"].values:
        print(f"\nERROR: El fármaco '{ejemplo_farmaco}' no existe en DrugBank limpio.")
        exit()

    print(f"\nProbando con fármaco: {ejemplo_farmaco}")

    dianas = set(df_drug[df_drug["DrugBank_ID"] == ejemplo_farmaco]["UniProt_ID"])
    dianas = list(dianas.intersection(G.nodes()))

    print(f"Dianas del fármaco presentes en red: {len(dianas)}")
    print(f"Dianas válidas: {dianas}")

    if len(dianas) == 0:
        print("\nERROR: El fármaco no tiene dianas presentes en el interactoma.")
        exit()

    d_obs, mu, sd, z, p = proximidad_estadistica(dianas, dist_ref, grados,bins,repeticiones=200)

    print(f"Distancia observada: {d_obs}")
    print(f"Media nula: {mu}")
    print(f"Desv. típica nula: {sd}")
    print(f"Z-score: {z}")
    print(f"P-valor: {p}")


Cargando BioGRID...
Cargando DrugBank limpio...
Cargando Enfermedades...
Genes en el CSV: ['CHN1', 'CHN', 'ARHGAP2', 'RHOGAP2', 'DURS2']
Genes convertidos a UniProt: 88
Proteínas de enfermedad presentes en red: 5
Proteínas válidas: ['P15882', 'P25189', 'P15882', 'P15882', 'P15882']

Probando con fármaco: DB00157
Dianas del fármaco presentes en red: 144
Dianas válidas: ['P15428', 'P03923', 'O43175', 'Q04828', 'P11586', 'Q16798', 'P00374', 'O43677', 'Q16836', 'P29803', 'P56937', 'Q15738', 'Q16795', 'P13995', 'P48448', 'P56556', 'P42330', 'Q9UI09', 'P04035', 'P16083', 'P09417', 'P00390', 'P03915', 'O60701', 'P53004', 'P30038', 'P08319', 'Q13630', 'P00367', 'P12268', 'O95168', 'P21695', 'P07864', 'Q9P0J0', 'P00325', 'P17516', 'P23368', 'Q9NRX3', 'O43181', 'O95167', 'O43920', 'P47895', 'P14679', 'Q08426', 'O75306', 'P05091', 'Q86Y39', 'P03886', 'P51649', 'P40925', 'O75489', 'P30519', 'P20839', 'O00483', 'P03897', 'O43837', 'O75438', 'P00387', 'O43674', 'P05093', 'P49448', 'Q92781', 'Q96C36'